In [1]:
import torch
import time
import datetime
import random
import numpy as np
import polars as pl
from scipy.sparse import csr_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import train_test_split
import seaborn as sns

In [2]:
EPOCHS = 10
BATCH_SIZE = 100

In [3]:
torch.set_default_dtype(torch.float32)
if torch.cuda.is_available():
    torch.set_default_device(0)
    print("Running on the GPU")
else:
    print("Running on the CPU")

Running on the CPU


[bood_id_map.csv](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/book_id_map.csv)
[user_id_map.csv](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/user_id_map.csv)
[goodreads_books.json.gz](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_books.json.gz)
[goodreads_reviews_dedup.json.gz](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_reviews_dedup.json.gz)
[goodreads_interactions.csv](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_interactions.csv)
[goodreads_interactions_dedup.json.gz](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_interactions_dedup.json.gz)

In [4]:
# book_ids = pl.read_csv('./data/book_id_map.csv')
# user_ids = pl.read_csv('./data/user_id_map.csv')
# books = pl.read_ndjson('./data/goodreads_books.json')
# reviews = pl.read_ndjson('./data/goodreads_reviews_dedup.json')
# interactions = pl.read_csv('./data/goodreads_interactions.csv')
# interactions_dedup = pl.read_ndjson('./data/goodreads_interactions_dedup.json')

# book_ids.write_ipc('./data/book_id_map.feather')
# user_ids.write_ipc('./data/user_id_map.feather')
# books.write_ipc('./data/goodreads_books.feather')
# reviews.write_ipc('./data/goodreads_reviews_dedup.feather')
# interactions.write_ipc('./data/goodreads_interactions.feather')
# interactions_dedup.write_ipc('./data/goodreads_interactions_dedup.feather')

In [5]:
book_ids = pl.read_ipc('./data/book_id_map.feather')
user_ids = pl.read_ipc('./data/user_id_map.feather')
books = pl.read_ipc('./data/goodreads_books.feather')
reviews = pl.read_ipc('./data/goodreads_reviews_dedup.feather')
interactions = pl.read_ipc('./data/goodreads_interactions.feather')
interactions_dedup = pl.read_ipc('./data/goodreads_interactions_dedup.feather')

In [6]:
book_ids.head()

book_id_csv,book_id
i64,i64
0,34684622
1,34536488
2,34017076
3,71730
4,30422361


In [7]:
book_ids.collect_schema()

Schema([('book_id_csv', Int64), ('book_id', Int64)])

In [8]:
user_ids.head()

user_id_csv,user_id
i64,str
0,"""8842281e1d1347389f2ab93d60773d…"
1,"""72fb0d0087d28c832f15776b0d9365…"
2,"""ab2923b738ea3082f5f3efcbbfacb2…"
3,"""d986f354a045ffb91234e4af4d1b12…"
4,"""7504b2aee1ecb5b2872d3da381c6c9…"


In [9]:
user_ids.collect_schema()

Schema([('user_id_csv', Int64), ('user_id', String)])

In [10]:
books.head()

isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,authors,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
str,str,list[str],str,str,list[struct[2]],str,str,str,str,list[str],str,str,str,list[struct[2]],str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""0312853122""","""1""",[],"""US""","""""","[{""3"",""to-read""}, {""1"",""p""}, … {""1"",""biography""}]","""""","""false""","""4.00""","""""",[],"""""","""Paperback""","""https://www.goodreads.com/book…","[{""604031"",""""}]","""St. Martin's Press""","""256""","""1""","""9780312853129""","""9""","""""","""1984""","""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…","""5333265""","""3""","""5400751""","""W.C. Fields: A Life on Film""","""W.C. Fields: A Life on Film"""
"""0743509986""","""6""",[],"""US""","""""","[{""2634"",""to-read""}, {""160"",""fiction""}, … {""2"",""general""}]","""""","""false""","""3.23""","""B000FC0PBC""","[""8709549"", ""17074050"", … ""307575""]","""Anita Diamant's international …","""Audio CD""","""https://www.goodreads.com/book…","[{""626222"",""""}]","""Simon & Schuster Audio""","""""","""1""","""9780743509985""","""10""","""Abridged""","""2001""","""https://www.goodreads.com/book…","""https://s.gr-assets.com/assets…","""1333909""","""10""","""1323437""","""Good Harbor""","""Good Harbor"""
"""""","""7""","[""189911""]","""US""","""eng""","[{""58"",""to-read""}, {""15"",""fantasy""}, … {""1"",""read-in-my-20s""}]","""B00071IKUY""","""false""","""4.03""","""""","[""19997"", ""828466"", … ""1597281""]","""Omnibus book club edition cont…","""Hardcover""","""https://www.goodreads.com/book…","[{""10333"",""""}]","""Nelson Doubleday, Inc.""","""600""","""""","""""","""""","""Book Club Edition""","""1987""","""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…","""7327624""","""140""","""8948723""","""The Unschooled Wizard (Sun Wol…","""The Unschooled Wizard (Sun Wol…"
"""0743294297""","""3282""",[],"""US""","""eng""","[{""7615"",""to-read""}, {""728"",""chick-lit""}, … {""6"",""bookworm-bitches""}]","""""","""false""","""3.49""","""B002ENBLOK""","[""6604176"", ""6054190"", … ""3134684""]","""Addie Downs and Valerie Adler …","""Hardcover""","""https://www.goodreads.com/book…","[{""9212"",""""}]","""Atria Books""","""368""","""14""","""9780743294294""","""7""","""""","""2009""","""https://www.goodreads.com/book…","""https://s.gr-assets.com/assets…","""6066819""","""51184""","""6243154""","""Best Friends Forever""","""Best Friends Forever"""
"""0850308712""","""5""",[],"""US""","""""","[{""32"",""to-read""}, {""3"",""runes""}, … {""1"",""magick-and-myth""}]","""""","""false""","""3.40""","""""",[],"""""","""""","""https://www.goodreads.com/book…","[{""149918"",""""}]","""""","""""","""""","""9780850308716""","""""","""""","""""","""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…","""287140""","""15""","""278577""","""Runic Astrology: Starcraft and…","""Runic Astrology: Starcraft and…"


- ~~('isbn', String)~~
- ('text_reviews_count', String) -> Integer
- ~~('series', List(String))~~
- ('country_code', String) -> Boolean
- ('language_code', String) -> One Hot Encoding
- ~~('popular_shelves', List(Struct({'count': String, 'name': String})))~~
- ~~('asin', String)~~
- ('is_ebook', String) -> Boolean
- ('average_rating', String) -> Float
- ~~('kindle_asin', String)~~
- ~~('similar_books', List(String))~~
- ~~('description', String)~~
- ('format', String) -> One Hot Encoding
- ~~('link', String)~~
- ~~('authors', List(Struct({'author_id': String, 'role': String})))~~
- ~~('publisher', String)~~
- ('num_pages', String) -> Integer / Float
- ~~('publication_day', String)~~
- ~~('isbn13', String)~~
- ~~('publication_month', String)~~
- ~~('edition_information', String)~~
- ('publication_year', String) -> Integer / Float
- ~~('url', String)~~
- ~~('image_url', String)~~
- ~~('book_id', String)~~
- ('ratings_count', String) -> Integer / Float
- ~~('work_id', String)~~
- ~~('title', String)~~
- ~~('title_without_series', String)~~

In [11]:
books.drop_in_place('isbn')
books.drop_in_place('series')
books.drop_in_place('popular_shelves')
books.drop_in_place('asin')
books.drop_in_place('kindle_asin')
books.drop_in_place('similar_books')
books.drop_in_place('description')
books.drop_in_place('link')
books.drop_in_place('authors')
books.drop_in_place('publisher')
books.drop_in_place('isbn13')
books.drop_in_place('edition_information')
books.drop_in_place('url')
books.drop_in_place('image_url')
books.drop_in_place('work_id')
books.drop_in_place('title')
books.drop_in_place('title_without_series')
books.drop_in_place('publication_month')
books.drop_in_place('publication_day')

publication_day
str
"""1"""
"""1"""
""""""
"""14"""
""""""
…
"""16"""
"""26"""
"""1"""


In [12]:
books = books.with_columns(pl.col('book_id').cast(pl.Int64))
books = books.filter(pl.col('publication_year').ne(''))
books = books.filter(pl.col('publication_year').cast(pl.Int64).le(2026))

In [13]:
# books = books.lazy().with_columns(
#     pl.when(pl.col('publication_month').str.len_chars().eq(0)).then(pl.col('publication_month').str.join('1')).otherwise(pl.col('publication_month')).alias('publication_month')
# ).collect(engine='streaming')

In [14]:
# books = books.lazy().with_columns(
#     pl.when(pl.col('publication_day').str.len_chars().eq(0)).then(pl.col('publication_day').str.join('1')).otherwise(pl.col('publication_day')).alias('publication_day')
# ).collect(engine='streaming')

In [15]:
# books = books.lazy().with_columns(
#     pl.when(pl.col('publication_year').str.starts_with('200') & pl.col('publication_year').str.len_chars().eq(5)).then(pl.col('publication_year').cast(pl.Int64).sub(18000).cast(pl.String)).otherwise(pl.col('publication_year')).alias('publication_year') # 12 fixes
# ).collect(engine='streaming')

In [16]:
# books = books.lazy().with_columns(
#     pl.when((pl.col('publication_year').str.starts_with('201') | pl.col('publication_year').str.starts_with('19')) & pl.col('publication_year').str.len_chars().eq(5)).then(pl.col('publication_year').str.slice(0,4)).otherwise(pl.col('publication_year')).alias('publication_year') # 42 fixes
# ).collect(engine='streaming')

In [17]:
# books = books.lazy().with_columns(
#     pl.when(pl.col('publication_year').str.starts_with('210') & pl.col('publication_year').str.len_chars().eq(5)).then(pl.col('publication_year').cast(pl.Int64).sub(19000).cast(pl.String)).otherwise(pl.col('publication_year')).alias('publication_year') # 5 fixes
# ).collect(engine='streaming')

In [18]:
# books = books.lazy().with_columns(
#     pl.when(pl.col('publication_year').str.starts_with('220') & pl.col('publication_year').str.len_chars().eq(5)).then(pl.col('publication_year').cast(pl.Int64).sub(20000).cast(pl.String)).otherwise(pl.col('publication_year')).alias('publication_year') # 6 fixes
# ).collect(engine='streaming')

In [19]:
# books = books.lazy().with_columns(
#     pl.when(pl.col('publication_year').str.starts_with('210') & pl.col('publication_year').str.len_chars().eq(4)).then(pl.col('publication_year').cast(pl.Int64).sub(90).cast(pl.String)).otherwise(pl.col('publication_year')).alias('publication_year') # 8 fixes
# ).collect(engine='streaming')

In [20]:
# books = books.lazy().with_columns(
#     pl.when(pl.col('publication_year').str.starts_with('30') & pl.col('publication_year').str.len_chars().eq(4)).then(pl.col('publication_year').cast(pl.Int64).sub(1000).cast(pl.String)).otherwise(pl.col('publication_year')).alias('publication_year') # 8 fixes
# ).collect(engine='streaming')

In [21]:
# books = books.lazy().with_columns(
#     pl.when(pl.col('publication_year').str.starts_with('40') & pl.col('publication_year').str.len_chars().eq(4)).then(pl.col('publication_year').cast(pl.Int64).sub(2000).cast(pl.String)).otherwise(pl.col('publication_year')).alias('publication_year') # 3 fixes
# ).collect(engine='streaming')

In [22]:
# books = books.lazy().with_columns(
#     pl.when((pl.col('publication_month').cast(pl.Int64).eq(2) | pl.col('publication_month').cast(pl.Int64).eq(4) | pl.col('publication_month').cast(pl.Int64).eq(6) | pl.col('publication_month').cast(pl.Int64).eq(9) | pl.col('publication_month').cast(pl.Int64).eq(11)) & pl.col('publication_day').cast(pl.Int64).ge(29)).then(pl.col('publication_day').str.join('28').slice(-2)).otherwise(pl.col('publication_day')).alias('publication_day')
# ).collect(engine='streaming')

In [23]:
# books = books.filter(pl.col('publication_year').cast(pl.Int64).le(2026))

In [24]:
# books = books.with_columns(publication_date=(
#                             pl.when(pl.col('publication_year').eq('')).then(1).otherwise(pl.col('publication_year')) + '-' + 
#                             pl.when(pl.col('publication_month').eq('')).then(1).otherwise(pl.col('publication_month')) + '-' +
#                             pl.when(pl.col('publication_day').eq('')).then(1).otherwise(pl.col('publication_day'))).str.to_date('%Y-%m-%d'))
# books.drop_in_place('publication_year')
# books.drop_in_place('publication_month')
# books.drop_in_place('publication_day')
# books.head()

In [25]:
# print(f'# Formats: {len(set(books['format']))}')
# print(f'Formats: {set(books['format'])}')
# print(f'# Publishers: {len(set(books['publisher']))}')
# # print(f'Publishers: {set(books['publisher'])}')
# print(f'# Edition Informations: {len(set(books['edition_information']))}')
# # print(f'Edition Informations: {set(books['edition_information'])}')
# print(f'# Language Codes: {len(set(books['language_code']))}')
# print(f'Language Codes: {set(books['language_code'])}')
# print(f'# Country Codes: {len(set(books['country_code']))}')
# print(f'Country Codes: {set(books['country_code'])}')

In [26]:
books.collect_schema()

Schema([('text_reviews_count', String),
        ('country_code', String),
        ('language_code', String),
        ('is_ebook', String),
        ('average_rating', String),
        ('format', String),
        ('num_pages', String),
        ('publication_year', String),
        ('book_id', Int64),
        ('ratings_count', String)])

In [27]:
reviews.head()

user_id,book_id,review_id,rating,review_text,date_added,date_updated,read_at,started_at,n_votes,n_comments
str,str,str,i64,str,str,str,str,str,i64,i64
"""8842281e1d1347389f2ab93d60773d…","""24375664""","""5cd416f3efc3f944fce4ce2db2290d…",5,"""Mind blowingly cool. Best scie…","""Fri Aug 25 13:55:02 -0700 2017""","""Mon Oct 09 08:55:59 -0700 2017""","""Sat Oct 07 00:00:00 -0700 2017""","""Sat Aug 26 00:00:00 -0700 2017""",16,0
"""8842281e1d1347389f2ab93d60773d…","""18245960""","""dfdbb7b0eb5a7e4c26d59a937e2e5f…",5,"""This is a special book. It sta…","""Sun Jul 30 07:44:10 -0700 2017""","""Wed Aug 30 00:00:26 -0700 2017""","""Sat Aug 26 12:05:52 -0700 2017""","""Tue Aug 15 13:23:18 -0700 2017""",28,1
"""8842281e1d1347389f2ab93d60773d…","""6392944""","""5e212a62bced17b4dbe41150e5bb90…",3,"""I haven't read a fun mystery b…","""Mon Jul 24 02:48:17 -0700 2017""","""Sun Jul 30 09:28:03 -0700 2017""","""Tue Jul 25 00:00:00 -0700 2017""","""Mon Jul 24 00:00:00 -0700 2017""",6,0
"""8842281e1d1347389f2ab93d60773d…","""22078596""","""fdd13cad0695656be99828cd75d6eb…",4,"""Fun, fast paced, and disturbin…","""Mon Jul 24 02:33:09 -0700 2017""","""Sun Jul 30 10:23:54 -0700 2017""","""Sun Jul 30 15:42:05 -0700 2017""","""Tue Jul 25 00:00:00 -0700 2017""",22,4
"""8842281e1d1347389f2ab93d60773d…","""6644782""","""bd0df91c9d918c0e433b9ab3a9a5c4…",4,"""A fun book that gives you a se…","""Mon Jul 24 02:28:14 -0700 2017""","""Thu Aug 24 00:07:20 -0700 2017""","""Sat Aug 05 00:00:00 -0700 2017""","""Sun Jul 30 00:00:00 -0700 2017""",8,0


In [28]:
reviews.collect_schema()

Schema([('user_id', String),
        ('book_id', String),
        ('review_id', String),
        ('rating', Int64),
        ('review_text', String),
        ('date_added', String),
        ('date_updated', String),
        ('read_at', String),
        ('started_at', String),
        ('n_votes', Int64),
        ('n_comments', Int64)])

In [29]:
interactions.head()

user_id,book_id,is_read,rating,is_reviewed
i64,i64,i64,i64,i64
0,948,1,5,0
0,947,1,5,1
0,946,1,5,0
0,945,1,5,0
0,944,1,5,0


In [30]:
interactions.collect_schema()

Schema([('user_id', Int64),
        ('book_id', Int64),
        ('is_read', Int64),
        ('rating', Int64),
        ('is_reviewed', Int64)])

In [31]:
interactions_dedup.head()

user_id,book_id,review_id,is_read,rating,review_text_incomplete,date_added,date_updated,read_at,started_at
str,str,str,bool,i64,str,str,str,str,str
"""8842281e1d1347389f2ab93d60773d…","""34684622""","""a53868823f065a0e20fd4ae98b8206…",false,0,"""""","""Tue Oct 17 09:40:11 -0700 2017""","""Tue Oct 17 09:40:12 -0700 2017""","""""",""""""
"""8842281e1d1347389f2ab93d60773d…","""34536488""","""9f08c5f991f87f3b7ae4ce779c2aac…",false,0,"""""","""Fri Oct 13 07:19:50 -0700 2017""","""Fri Oct 13 07:19:50 -0700 2017""","""""",""""""
"""8842281e1d1347389f2ab93d60773d…","""34017076""","""14da595c5b0c38b1247888f62f74a7…",false,0,"""""","""Fri Oct 06 09:32:42 -0700 2017""","""Fri Oct 06 09:32:43 -0700 2017""","""""",""""""
"""8842281e1d1347389f2ab93d60773d…","""71730""","""bd1a29916eb3ea1d0d45d4f7395920…",false,0,"""""","""Tue Oct 03 23:31:03 -0700 2017""","""Tue Oct 03 23:39:42 -0700 2017""","""""",""""""
"""8842281e1d1347389f2ab93d60773d…","""30422361""","""d2edf29508fd36808bf30b37043acd…",false,0,"""""","""Tue Oct 03 15:05:28 -0700 2017""","""Tue Oct 03 15:05:29 -0700 2017""","""""",""""""


In [32]:
print(f'Ratings: {set(interactions_dedup['rating'])}')

Ratings: {0, 1, 2, 3, 4, 5}


In [33]:
interactions_dedup.collect_schema()

Schema([('user_id', String),
        ('book_id', String),
        ('review_id', String),
        ('is_read', Boolean),
        ('rating', Int64),
        ('review_text_incomplete', String),
        ('date_added', String),
        ('date_updated', String),
        ('read_at', String),
        ('started_at', String)])

In [34]:
interactions = interactions.filter(pl.col('is_read').eq(1))
interactions = interactions.drop('is_read')
interactions = interactions.filter(pl.col('user_id').is_duplicated())

In [35]:
print(len(interactions))

112101948


In [36]:
print(len(books))

1760536


In [37]:
books.collect_schema()

Schema([('text_reviews_count', String),
        ('country_code', String),
        ('language_code', String),
        ('is_ebook', String),
        ('average_rating', String),
        ('format', String),
        ('num_pages', String),
        ('publication_year', String),
        ('book_id', Int64),
        ('ratings_count', String)])

In [38]:
books['text_reviews_count'].fill_null(0)

text_reviews_count
str
"""1"""
"""6"""
"""7"""
"""3282"""
"""7"""
…
"""2"""
"""3"""
"""2"""


In [39]:
books = books.with_columns(pl.col('text_reviews_count').str.replace('', '0').cast(pl.Float32))
books = books.with_columns(pl.col('is_ebook').replace('false', '0').replace('true', '1').cast(pl.Float32))
books = books.with_columns(pl.col('average_rating').str.replace('', '0').cast(pl.Float32))
books = books.with_columns(pl.col('num_pages').str.replace('', '0').cast(pl.Float32))
books = books.with_columns(pl.col('publication_year').cast(pl.Float32))
books = books.with_columns(pl.col('ratings_count').replace('', '0').cast(pl.Float32))
books = books.with_columns(pl.col('country_code').replace('', '0').replace('US', '1').cast(pl.Float32))
books = books.to_dummies('language_code')
books = books.to_dummies('format')

In [40]:
users = books.join(interactions, on='book_id')

In [41]:
users.group_by('user_id').agg(pl.exclude('book_id').mean())

user_id,text_reviews_count,country_code,language_code_,language_code_--,language_code_abk,language_code_ace,language_code_ach,language_code_ada,language_code_ady,language_code_afa,language_code_afr,language_code_aka,language_code_ale,language_code_alg,language_code_amh,language_code_ang,language_code_anp,language_code_apa,language_code_ara,language_code_arg,language_code_arp,language_code_arw,language_code_asm,language_code_ast,language_code_aus,language_code_ava,language_code_ave,language_code_aze,language_code_bam,language_code_bas,language_code_bat,language_code_bel,language_code_ben,language_code_bik,language_code_bos,language_code_btk,…,"format_trade paperback, e-book","format_trade, paperback","format_true first ed HC, re-bound by library",format_trykhy,format_turtleback,"format_twarda (hardcover), 145 x 205 mm","format_txt, pdf, etc.",format_unabridged audio download,format_unbound cards w/ envelope,format_unknown,format_unproofed e-ARC,format_urdu,format_vazana vazba,format_viii+490 pages,format_visual novel,format_wattpad,format_wattpad book,"format_wattpad, pdf.txt",format_web,format_web novel,format_webpage,format_website,format_webtoon,format_wee book,format_word,format_words,format_wraps; un-cut,format_wrq Glf `dy,format_wrqy,format_wzyry,format_zine,format_zip,num_pages,publication_year,ratings_count,rating,is_reviewed
i64,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,f64,f64
159660,643.915283,1.0,0.440678,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,290.889832,1998.610229,28164.017578,3.440678,0.135593
505737,42.0,1.0,0.363636,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,347.272736,2000.363647,1046.0,3.636364,0.0
138033,216.320007,1.0,0.56,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,215.399994,2000.47998,5189.799805,3.88,0.8
759613,208.857147,1.0,0.285714,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,211.142853,2000.857178,2449.428467,4.0,0.285714
721343,81.5,1.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,284.0,1986.0,639.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
255716,258.13147,1.0,0.505976,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,300.286865,1999.717163,9091.956055,3.430279,0.007968
89518,602.521729,1.0,0.434783,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0

In [42]:
print(len(users.columns))

2302


In [43]:
def normalize_ids(df: pl.DataFrame, id: str):
    uniq_ids = set(df[id].unique())
    ids_to_indices = { orig_id : new_id for new_id, orig_id in enumerate(uniq_ids) }
    new_ids = df.with_columns(pl.col(id).replace(ids_to_indices).cast(pl.Int64))
    indices_to_ids = { new_id : orig_id for orig_id, new_id in ids_to_indices.items()}
    return new_ids, indices_to_ids

In [44]:
users, _ = normalize_ids(users, 'user_id')

users_n_books = users['book_id'].n_unique()

In [ ]:
gss = GroupShuffleSplit(n_splits=5)

users_train_idx, users_test_idx = next(gss.split(X=users, groups=users['user_id']))
users_train: pl.Dataframe = users[users_train_idx]
users_test:  pl.Dataframe = users[users_test_idx]

users_train, users_train_map_id = normalize_ids(users_train, 'user_id')
users_test,  users_test_map_id  = normalize_ids(users_test, 'user_id')

# users_train_csr = csr_matrix((users_train['rating'], (users_train['user_id'], users_train['book_id'])), shape=(users_train['user_id'].n_unique(), users_n_books))

In [ ]:
users_test = users_test.filter(pl.col('user_id').is_duplicated())
users_train = users_train.filter(pl.col('user_id').is_in(users_test['user_id'].implode()))

In [ ]:
users_test_seen,   users_test_unseen         = train_test_split(users_test, train_size=0.7, stratify=users_test['user_id'])

users_test_seen,   users_test_seen_map_id    = normalize_ids(users_test_seen,   'user_id')
users_test_unseen, users_test_unseen_map_id  = normalize_ids(users_test_unseen, 'user_id')

# users_test_seen_csr = csr_matrix((users_test_seen['rating'], (users_test_seen['user_id'], users_test_seen['book_id'])), shape=(users_test_seen['user_id'].n_unique(), users_n_books))

In [ ]:
users_train = users_train.group_by('user_id')
users_test_seen = users_test_seen.group_by('user_id').agg(pl.exclude('book_id').mean())
users_test_unseen = users_test_unseen.group_by('user_id').agg(pl.exclude('book_id').mean())

In [ ]:
# train_crow_idx = interactions_train_csr.tocoo().coords[0]
# np.append(train_crow_idx, interactions_train_csr.getnnz(axis=0))
# train_col_idx = interactions_train_csr.tocoo().coords[1]
# np.append(train_crow_idx, interactions_train_csr.getnnz(axis=1))
# train_values = interactions_train_csr.tocoo().data

# test_crow_idx = interactions_test_seen_csr.tocoo().coords[0]
# test_col_idx = interactions_test_seen_csr.tocoo().coords[1]
# test_values = interactions_test_seen_csr.tocoo().data

# interactions_features = torch.sparse_csr_tensor(crow_indices=torch.tensor(train_crow_idx, dtype=torch.int64),
#                                                 col_indices=torch.tensor(train_col_idx, dtype=torch.int64),
#                                                 values=torch.tensor(train_values, dtype=torch.int64))
# interactions_labels = torch.sparse_csr_tensor(crow_indices=torch.tensor(test_crow_idx, dtype=torch.int64),
#                                               col_indices=torch.tensor(test_col_idx, dtype=torch.int64),
#                                               values=torch.tensor(test_values, dtype=torch.int64))

In [ ]:
class Timer(object):
    def __init__(self, name=None, filename=None):
        self.name = name
        self.filename = filename

    def __enter__(self):
        self.tstart = time.time()

    def __exit__(self, type, value, traceback):
        message = 'Elapsed: %.2f seconds' % (time.time() - self.tstart)
        if self.name:
            message = '[%s] ' % self.name + message
        print(message)
        if self.filename:
            with open(self.filename,'a') as file:
                print(str(datetime.datetime.now())+": ",message,file=file)

In [ ]:
# print(interactions['user_id'].n_unique())
# print(interactions_n_books)
# print(interactions_train_csr.nnz)

In [ ]:
class AE(torch.nn.Module):
    def __init__(self):
        super(AE, self).__init__()
        self.model = torch.nn.Sequential()
        self.model.append(torch.nn.Linear(2300,1150))
        self.model.append(torch.nn.ReLU())
        self.model.append(torch.nn.Linear(1150,1150))
        self.model.append(torch.nn.ReLU())
        self.model.append(torch.nn.Linear(1150,2300))
        self.model.append(torch.nn.Sigmoid())
        
    def forward(self, x):
        return self.model(x)

In [ ]:
#loader = torch.utils.data.DataLoader(dataset=interactions_train_csr.data, batch_size=BATCH_SIZE, shuffle=True)
auto_encoder = AE()
loss_function = torch.nn.MSELoss()
optimizer = torch.optim.Adam(auto_encoder.parameters(), lr=0.001, weight_decay=0.00000001)

In [ ]:
training_loss = []

In [ ]:
with Timer('Total time'):
    with Timer('Training time'):
        for epoch in range(EPOCHS):
            for name, data in users_train:
                x = data.drop('user_id').drop('book_id').to_torch(dtype=pl.Float32)
                y = users_test_seen.filter(pl.col('user_id').eq(name[0])).drop('user_id').to_torch(dtype=pl.Float32)
                pred = auto_encoder(x).mean(dim=0, keepdim=True)
                loss = loss_function(pred, y)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                training_loss.append(loss.item())

In [ ]:
sns.lineplot(training_loss)